<a href="https://colab.research.google.com/github/Andrey66Sudakov/web_scraping/blob/main/web_scraping_task_Sudakov.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Установка библиотек
!pip install requests beautifulsoup4 pandas

In [7]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def search_quotes_by_text(query_list):
    """
    Ищет цитаты, в тексте которых есть хотя бы одно слово из query_list.
    Используется ТОЛЬКО первая страница результатов (как в задании).
    """
    url = "https://quotes.toscrape.com/page/1/"
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')

    quotes = soup.find_all('div', class_='quote')
    data_list = []
    seen_texts = set()

    for quote in quotes:
        text_span = quote.find('span', class_='text')
        quote_text = text_span.get_text(strip=True) if text_span else ""

        # Проверяем, есть ли хоть одно из поисковых слов в тексте цитаты
        lower_text = quote_text.lower()
        if any(query.lower() in lower_text for query in query_list):
            if quote_text not in seen_texts:  # убираем дубликаты по тексту
                seen_texts.add(quote_text)

                # Имитируем "дату" - возьмём автора, как будто это дата публикации
                # (или можно просто поставить пустую строку)
                author = quote.find('small', class_='author')
                author_name = author.get_text(strip=True) if author else ""

                link_a = quote.find('a', href=True)
                full_link = "https://quotes.toscrape.com" + link_a['href'] if link_a else ""

                data_list.append({
                    'Дата (автор)': author_name,  # вынужденная замена
                    'Заголовок (текст цитаты)': quote_text,
                    'Ссылка': full_link
                })

    return pd.DataFrame(data_list)

# --- ЗАВЕРШЕНИЕ ФУНКЦИИ ---

In [10]:
# Задаем список "поисковых запросов" (тегов)
search_queries = ['love', 'inspirational', 'humor']

# Вызываем функцию
result_df = scrape_quotes_by_tags(search_queries)

# Выводим результат
if not result_df.empty:
    print(f"Найдено записей: {len(result_df)}")
    display(result_df.head(15))  # Показываем первые 15 строк
else:
    print("Данные не найдены или произошла ошибка.")

Найдено записей: 24


,Автор (вместо даты),Цитата (вместо заголовка),Ссылка на материал
0,André Gide,“It is better to be hated for what you are tha...,https://quotes.toscrape.com/author/Andre-Gide
1,Marilyn Monroe,“This life is what you make it. No matter what...,https://quotes.toscrape.com/author/Marilyn-Monroe
2,Bob Marley,"“You may not be her first, her last, or her on...",https://quotes.toscrape.com/author/Bob-Marley
3,Elie Wiesel,"“The opposite of love is not hate, it's indiff...",https://quotes.toscrape.com/author/Elie-Wiesel
4,Friedrich Nietzsche,"“It is not a lack of love, but a lack of frien...",https://quotes.toscrape.com/author/Friedrich-N...
5,Pablo Neruda,"“I love you without knowing how, or when, or f...",https://quotes.toscrape.com/author/Pablo-Neruda
6,James Baldwin,“Love does not begin and end the way we seem t...,https://quotes.toscrape.com/author/James-Baldwin
7,Jane Austen,“There is nothing I would not do for those who...,https://quotes.toscrape.com/author/Jane-Austen
8,Albert Einstein,“There are only two ways to live your life. On...,https://quotes.toscrape.com/author/Albert-Eins...
9,Thomas A. Edison,"“I have not failed. I've just found 10,000 way...",https://quotes.toscrape.com/author/Thomas-A-Ed...


На habr.com я бы искал заголовки статей в тегах (h2 class="post__title") или (a class="post__title_link"), а дату — в (span class="post__time")

Поскольку habr постоянно меняет верстку, я заменил дату на имя автора (в учебных целях)